# Local dataset audit for supervision meetings

This notebook reads the ignored local OptionMetrics extracts. Run it only on an
authorised machine. Before committing, keep every code-cell output cleared.

# 导师会议本地数据检查

本 Notebook 读取已被 Git 忽略的本地 OptionMetrics 文件。仅应在获得授权的
设备上运行；提交前必须清除全部代码单元输出。

In [ ]:
from pathlib import Path
import csv
import itertools
import json

try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    class Markdown(str):
        pass

    def display(value):
        print(str(value))


def find_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


ROOT = find_repository_root()
DATA_DIR = ROOT / Path("data")
PROFILE_PATH = ROOT / "docs" / "data" / "dataset_profile.json"
EXPECTED_FILES = [
    "Index Dividend Yield.csv",
    "Zero Coupon Yield Curve.csv",
    "option price.csv",
    "security price.csv",
]

## Local file check / 本地文件检查

This cell reports file presence and size without reading individual rows.

本单元仅报告文件是否存在及文件大小，不读取单条记录。

In [ ]:
file_status = []
for name in EXPECTED_FILES:
    path = DATA_DIR / name
    file_status.append({
        "file": name,
        "present": path.is_file(),
        "size_gib": round(path.stat().st_size / (1024 ** 3), 3) if path.is_file() else None,
    })
file_status

## Temporary five-row preview / 临时五行预览

This output may contain licensed observations. Use it during the meeting and
clear the notebook output before saving or committing.

该输出可能包含受许可保护的真实观测。仅在会议中临时展示，保存或提交前必须清除。

In [ ]:
preview = {}
for name in EXPECTED_FILES:
    path = DATA_DIR / name
    if not path.is_file():
        preview[name] = {"error": "missing local file"}
        continue
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        preview[name] = list(itertools.islice(reader, 5))
preview

## Aggregate profile / 汇总概况

Generate or refresh the aggregate profile before the meeting:

```bash
python3 scripts/profile_datasets.py   --data-dir data   --output docs/data/dataset_profile.json   --batch-size 100000
```

In [ ]:
if not PROFILE_PATH.is_file():
    display(Markdown("Run the profiling command above before the meeting."))
else:
    profile = json.loads(PROFILE_PATH.read_text(encoding="utf-8"))
    summary = {
        name: {
            "rows": table["row_count"],
            "date_range": table["date_range"],
            "columns": len(table["columns"]),
        }
        for name, table in profile["tables"].items()
    }
    summary

## Before committing / 提交前

Use **Kernel → Restart & Clear Output**, save the notebook, and confirm that
every code cell has an empty `outputs` list.

请选择 **Kernel → Restart & Clear Output**，保存后确认所有代码单元的
`outputs` 均为空。